# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Srinadh2314/srinadh-flyrank-intership/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.


## 1. Ranked actions + reason codes

The model produces a ranked review queue rather than an automatic content decision. The highest-ranked items are marked `REVIEW_FIRST` because they are the strongest candidates for human inspection under the selected outcome proxy.

Reason codes explain why an item entered the queue:
- `MODEL_HIGH_PRIORITY` — high model score and within the review set.
- `MODEL_LOWER_PRIORITY` — lower-ranked item outside the initial review set.

The reason code describes prioritization, not a guaranteed content action.

In [9]:
import pandas as pd
import numpy as np
import duckdb
from google.colab import userdata
from sklearn.ensemble import RandomForestClassifier


HF_TOKEN = userdata.get("HF_TOKEN")
print("HF_TOKEN loaded successfully")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
)
""")

print("Warehouse connection ready")


feature_query = """
SELECT
    client_hash_id,
    content_hash_id,
    AVG(gsc_impressions) AS avg_gsc_impressions,
    AVG(gsc_clicks) AS avg_gsc_clicks,
    AVG(gsc_avg_position) AS avg_gsc_avg_position,
    AVG(ga4_sessions) AS avg_ga4_sessions,
    AVG(scroll_events) AS avg_scroll_events
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/*.parquet'
)
GROUP BY client_hash_id, content_hash_id
"""

feature_df = con.execute(feature_query).df()


label_query = """
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS march_impressions
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
GROUP BY client_hash_id, content_hash_id
"""

label_df = con.execute(label_query).df()


model_df = feature_df.merge(
    label_df,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

model_df["target"] = (
    model_df["march_impressions"] == 0
).astype(int)

features = [
    "avg_gsc_impressions",
    "avg_gsc_clicks",
    "avg_gsc_avg_position",
    "avg_ga4_sessions",
    "avg_scroll_events"
]

X = (
    model_df[features]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

y = model_df["target"]


from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        X,
        y,
        groups=model_df["client_hash_id"]
    )
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

model_score = model.predict_proba(X_test)[:, 1]


recommendations = model_df.iloc[test_idx][
    ["client_hash_id", "content_hash_id"] + features
].copy()

recommendations["model_score"] = model_score

recommendations = recommendations.sort_values(
    "model_score",
    ascending=False
).reset_index(drop=True)

recommendations["rank"] = np.arange(
    1, len(recommendations) + 1
)

recommendations["action"] = np.where(
    recommendations["rank"] <= 50,
    "REVIEW_FIRST",
    "LOWER_PRIORITY"
)

recommendations["reason_code"] = np.where(
    recommendations["rank"] <= 50,
    "MODEL_HIGH_PRIORITY",
    "MODEL_LOWER_PRIORITY"
)

print("Total ranked rows:", len(recommendations))
print("Review-first rows:", (recommendations["rank"] <= 50).sum())

display(
    recommendations[
        [
            "rank",
            "model_score",
            "action",
            "reason_code"
        ]
    ].head(20)
)

HF_TOKEN loaded successfully
Warehouse connection ready


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total ranked rows: 39438
Review-first rows: 50


,rank,model_score,action,reason_code
0,1,0.963841,REVIEW_FIRST,MODEL_HIGH_PRIORITY
1,2,0.963841,REVIEW_FIRST,MODEL_HIGH_PRIORITY
2,3,0.963841,REVIEW_FIRST,MODEL_HIGH_PRIORITY
3,4,0.963841,REVIEW_FIRST,MODEL_HIGH_PRIORITY
4,5,0.963841,REVIEW_FIRST,MODEL_HIGH_PRIORITY
5,6,0.963841,REVIEW_FIRST,MODEL_HIGH_PRIORITY
6,7,0.963841,REVIEW_FIRST,MODEL_HIGH_PRIORITY
7,8,0.963841,REVIEW_FIRST,MODEL_HIGH_PRIORITY
8,9,0.951763,REVIEW_FIRST,MODEL_HIGH_PRIORITY
9,10,0.951763,REVIEW_FIRST,MODEL_HIGH_PRIORITY


## 2. Intended use and limits

This playbook is intended for content reviewers who need to prioritize a large set of pages for inspection.

The output is a decision-support queue, not an automatic content decision. A reviewer should inspect the page and its context before choosing an action.

The recommendations are specific to the selected outcome proxy, February 2026 feature window, model, and validation design. They should not be treated as guarantees of future search performance.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Intended user: Content reviewer")
print("Primary use: Prioritize pages for human review")
print("Automatic content action: No")
print("Decision-support only: Yes")

Intended user: Content reviewer
Primary use: Prioritize pages for human review
Automatic content action: No
Decision-support only: Yes


## 3. Human review + the no-go list

Before acting on a recommendation, a reviewer must inspect the page context, search intent, relevance, content quality, freshness, and business importance.

The model only provides prioritization evidence. A human must make the final content decision.

The following actions should never be automated from this model alone:
- publishing or deleting content
- declaring a page low quality
- guaranteeing ranking improvement
- making causal claims about search performance
- exposing private client or content information

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
no_go_items = [
    "Automatic content deletion",
    "Automatic publication",
    "Automatic quality judgment",
    "Guaranteeing ranking improvement",
    "Causal claims from this model"
]

no_go_table = pd.DataFrame({
    "no_go_action": no_go_items,
    "automate_from_model": [False] * len(no_go_items)
})

display(no_go_table)


,no_go_action,automate_from_model
0,Automatic content deletion,False
1,Automatic publication,False
2,Automatic quality judgment,False
3,Guaranteeing ranking improvement,False
4,Causal claims from this model,False



## 4. Monitoring / retrain triggers

The playbook should be reviewed when the underlying data, outcome definition, or client population changes.

Possible triggers include:
- a substantial change in feature distributions
- increased missingness in important signals
- a change in the outcome or label definition
- lower Precision@50 on a later evaluation window
- meaningful changes in the client population
- reviewer feedback that recommendations are no longer useful

These triggers indicate when the model should be re-evaluated or retrained rather than assuming the existing ranking remains valid.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
monitoring_triggers = pd.DataFrame({
    "trigger": [
        "Feature distribution shift",
        "Outcome/label definition changes",
        "Lower future Precision@50",
        "Higher feature missingness",
        "Client population changes",
        "Reviewer feedback indicates stale recommendations"
    ],
    "response": [
        "Re-evaluate",
        "Rebuild target and revalidate",
        "Investigate and retrain",
        "Investigate data availability",
        "Revalidate grouped split",
        "Review feature/model design"
    ]
})

display(monitoring_triggers)


,trigger,response
0,Feature distribution shift,Re-evaluate
1,Outcome/label definition changes,Rebuild target and revalidate
2,Lower future Precision@50,Investigate and retrain
3,Higher feature missingness,Investigate data availability
4,Client population changes,Revalidate grouped split
5,Reviewer feedback indicates stale recommendations,Review feature/model design



## 5. Exports for the paper

The notebook exports a privacy-safe ranked recommendation queue and a metrics summary for reuse in the final research paper.

Client and content identifiers are excluded from the paper-facing export. The exported files contain only ranking, model score, action, reason code, and summary metrics.

In [13]:
import os
import json
import pandas as pd
import numpy as np

os.makedirs("work/outputs", exist_ok=True)


evaluation_queue = model_df.iloc[test_idx][
    ["client_hash_id", "content_hash_id", "target"]
].copy()

evaluation_queue["model_score"] = model_score

evaluation_queue["original_order"] = np.arange(
    len(evaluation_queue)
)

evaluation_queue = evaluation_queue.sort_values(
    ["model_score", "original_order"],
    ascending=[False, True]
).reset_index(drop=True)

evaluation_queue["rank"] = np.arange(
    1, len(evaluation_queue) + 1
)

evaluation_queue["action"] = np.where(
    evaluation_queue["rank"] <= 50,
    "REVIEW_FIRST",
    "LOWER_PRIORITY"
)

evaluation_queue["reason_code"] = np.where(
    evaluation_queue["rank"] <= 50,
    "MODEL_HIGH_PRIORITY",
    "MODEL_LOWER_PRIORITY"
)


top50 = evaluation_queue.head(50)

rf_precision_at_50 = top50["target"].mean()

print("Top-50 positive rows:", int(top50["target"].sum()))
print(f"Precision@50: {rf_precision_at_50:.3f}")


paper_queue = evaluation_queue[
    [
        "rank",
        "model_score",
        "action",
        "reason_code"
    ]
].copy()

paper_queue.to_csv(
    "work/outputs/paper_ranked_recommendations.csv",
    index=False
)


metrics_receipt = {
    "random_forest_precision_at_50": float(rf_precision_at_50),
    "test_rows": int(len(test_idx)),
    "review_first_rows": 50
}

with open(
    "work/outputs/capstone_metrics.json",
    "w"
) as f:
    json.dump(
        metrics_receipt,
        f,
        indent=2
    )

print("\nSaved:")
print("work/outputs/paper_ranked_recommendations.csv")
print("work/outputs/capstone_metrics.json")

print("\nMetrics receipt:")
print(json.dumps(metrics_receipt, indent=2))

print("\nTop 10 paper-safe recommendations:")
display(paper_queue.head(10))

Top-50 positive rows: 48
Precision@50: 0.960

Saved:
work/outputs/paper_ranked_recommendations.csv
work/outputs/capstone_metrics.json

Metrics receipt:
{
  "random_forest_precision_at_50": 0.96,
  "test_rows": 39438,
  "review_first_rows": 50
}

Top 10 paper-safe recommendations:


,rank,model_score,action,reason_code
0,1,0.963841,REVIEW_FIRST,MODEL_HIGH_PRIORITY
1,2,0.963841,REVIEW_FIRST,MODEL_HIGH_PRIORITY
2,3,0.963841,REVIEW_FIRST,MODEL_HIGH_PRIORITY
3,4,0.963841,REVIEW_FIRST,MODEL_HIGH_PRIORITY
4,5,0.963841,REVIEW_FIRST,MODEL_HIGH_PRIORITY
5,6,0.963841,REVIEW_FIRST,MODEL_HIGH_PRIORITY
6,7,0.963841,REVIEW_FIRST,MODEL_HIGH_PRIORITY
7,8,0.963841,REVIEW_FIRST,MODEL_HIGH_PRIORITY
8,9,0.951763,REVIEW_FIRST,MODEL_HIGH_PRIORITY
9,10,0.951763,REVIEW_FIRST,MODEL_HIGH_PRIORITY


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.